# Model Evaluation

## Objectives
- Evaluate the trained CNN model
- Review learning curves for accuracy and loss
- Test model performance on unseen test data
- Confirm whether the model meets the 97% accuracy requirement

## Inputs
- Saved model from outputs/models/mildew_detector.h5
- Training history from outputs/models/training_history.csv
- Test dataset from outputs/datasets/test

## Outputs
- Accuracy and loss learning curves
- Test set accuracy
- Confusion matrix
- Classification report
- Final statement on whether the model meets the business requirement

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

In [ ]:
model = tf.keras.models.load_model("../outputs/models/mildew_detector.h5")
history_df = pd.read_csv("../outputs/models/training_history.csv")
history_df.head()

In [ ]:
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.savefig("../outputs/plots/accuracy_plot.png")
plt.show()
plt.close()

## Accuracy Learning Curve

The accuracy curve shows how the model improved during training and whether validation performance followed training performance.

In [ ]:
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("../outputs/plots/loss_plot.png")
plt.show()
plt.close()

## Loss Learning Curve

The loss curve helps identify whether the model is learning effectively and whether there are signs of overfitting.

In [ ]:
test_path = "../outputs/datasets/test"

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(100, 100),
    batch_size=20,
    class_mode="binary",
    shuffle=False
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
pred_probs = model.predict(test_data)
pred_classes = (pred_probs > 0.5).astype(int).reshape(-1)

true_classes = test_data.classes
class_labels = list(test_data.class_indices.keys())

class_labels

In [ ]:
cm = confusion_matrix(true_classes, pred_classes)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], class_labels)
plt.yticks([0, 1], class_labels)
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.savefig("../outputs/plots/confusion_matrix.png")
plt.show()
plt.close()

In [ ]:
print(classification_report(true_classes, pred_classes, target_names=class_labels))

## Model Evaluation Conclusion

The model achieved a test accuracy of 99%.

The agreed business requirement was a minimum of 97% accuracy.

Therefore, the model successfully meets and exceeds the business requirement.

This confirms that the model is effective in predicting whether a cherry leaf is healthy or affected by powdery mildew.